# SpatialVLA × LIBERO — Confidence 추출 분석

**목표**: SpatialVLA가 LIBERO 시뮬레이션 환경에서 gripper → target object 접근 시
confidence (action token log-probability)가 올라가는지 검증한다.

**방법**:
- LIBERO MuJoCo 시뮬레이션을 실제 실행
- 매 스텝마다 SpatialVLA inference → `generate(output_scores=True)` 로 logit 추출
- **confidence = Σ log P(translation token, rotation token, gripper token)**
- 로봇 EEF Z좌표 vs confidence 상관관계 계산

**요구사항**: Google Colab GPU 런타임 (T4 이상)

## Cell 0: 시스템 패키지 설치 (EGL 헤드리스 렌더링)

In [ ]:
%%bash
apt-get install -y libegl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
echo "EGL packages installed"

## Cell 1: pip 의존성 설치

> **중요**: 설치 완료 후 **Runtime → Restart session** 을 실행한 뒤 Cell 2부터 다시 실행하세요.

버전 핀 이유:
- `transformers==4.47.0`: 4.50+ 에서 `processing_spatialvla.py` import가 깨짐
- `robosuite==1.4.1`: 1.5+ 에서 `SingleArmEnv` 삭제 → LIBERO import 실패

In [ ]:
import torch
print(f"PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}")

# SpatialVLA 추론 의존성
!pip install "transformers==4.47.0" accelerate==1.0.1 peft==0.14.0 einops==0.8.0 scipy pillow -q

# MuJoCo + robosuite (버전 핀 필수)
!pip install mujoco "robosuite==1.4.1" -q

# LIBERO 소스 설치 (BDDL 파일 경로 자동 인식)
!git clone --quiet https://github.com/Lifelong-Robot-Learning/LIBERO.git
!pip install -e LIBERO -q

# robosuite macros 초기화 (없으면 ModuleNotFoundError)
!python -c "
import robosuite, os, subprocess, sys
setup = os.path.join(os.path.dirname(robosuite.__file__), 'scripts', 'setup_macros.py')
subprocess.run([sys.executable, setup])
print('macros initialized')
"

print("\n=== 설치 완료 ===")
print("Runtime → Restart session 후 Cell 2부터 다시 실행하세요")

## Cell 2: 환경변수 설정 + SpatialVLA 모델 로드

> `MUJOCO_GL=egl` 은 **어떤 import보다 먼저** 설정해야 합니다.

In [ ]:
import os
# Colab GPU (T4/A100) 헤드리스 서버 → EGL 렌더링 (import 전에 설정 필수)
os.environ["MUJOCO_GL"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoModel, AutoProcessor

MODEL_ID   = "IPEC-COMMUNITY/spatialvla-4b-224-sft-bridge"
UNNORM_KEY = "bridge_orig/1.0.0"  # bridge fine-tuned 모델의 action 정규화 키

print("SpatialVLA 모델 로딩 중 (약 2~3분)...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.bfloat16
).eval().cuda()
print(f"모델 로드 완료. device={next(model.parameters()).device}")

## Cell 3: LIBERO 환경 로드 + 에피소드 초기화

사용 가능한 task suite:
- `libero_spatial` : 공간적 배치 변형 (10 tasks)
- `libero_object`  : 다양한 오브젝트 (10 tasks)
- `libero_goal`    : 다양한 목표 (10 tasks)
- `libero_10`      : 긴 horizon (10 tasks)

In [ ]:
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

TASK_SUITE = "libero_spatial"  # 변경 가능
TASK_ID    = 0                 # 0~9 중 선택

task_suite = benchmark.get_benchmark_dict()[TASK_SUITE]()
task       = task_suite.get_task(TASK_ID)
task_desc  = task.language
print(f"Task: '{task_desc}'")
print(f"Available tasks: {task_suite.n_tasks}")

bddl_path = os.path.join(
    task_suite.get_task_bddl_file_dir(),
    task.problem_folder,
    task.bddl_file
)

env = OffScreenRenderEnv(
    bddl_file_name=bddl_path,
    camera_heights=256,
    camera_widths=256,
)
env.seed(0)
obs = env.set_init_state(task_suite.get_task_init_states(TASK_ID)[0])

# physics warm-up: 5 no-op steps
for _ in range(5):
    obs, _, _, _ = env.step([0.0] * 7)

print(f"Observation keys: {list(obs.keys())}")
print(f"Image shape: {obs['agentview_image'].shape}")

# 초기 프레임 미리보기
plt.figure(figsize=(5, 4))
plt.imshow(obs['agentview_image'][::-1, ::-1])  # 180도 회전해서 표시
plt.title(f"Initial observation\n'{task_desc}'")
plt.axis("off")
plt.show()

## Cell 4: 에피소드 실행 + Confidence 추출

핵심:
- `model.generate(output_scores=True)` → 각 action token의 logit 추출
- `predict_action()` 는 `output_scores` 를 지원하지 않아서 우회
- 이미지는 `[::-1, ::-1]` 로 반전 (LIBERO 렌더러가 상하좌우 반전된 이미지를 줌)

In [ ]:
def get_action_and_confidence(img_arr, text):
    """
    img_arr: (H, W, 3) uint8 numpy array, LIBERO agentview_image
    Returns: (action_7d, conf, [lp_trans, lp_rot, lp_grip])
    """
    # LIBERO 이미지는 상하좌우 반전 → SpatialVLA 학습 전처리에 맞게 보정
    image  = Image.fromarray(img_arr[::-1, ::-1].copy())
    inputs = processor(
        images=[image], text=text, unnorm_key=UNNORM_KEY, return_tensors="pt"
    ).to(model.device)
    inputs = {k: v.to(torch.bfloat16) if torch.is_floating_point(v) else v
              for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    # predict_action() 대신 generate() 직접 호출 → output_scores=True 로 logit 추출
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=3,               # translation, rotation, gripper 토큰
            return_dict_in_generate=True,
            output_scores=True,             # ← confidence 추출 핵심
            do_sample=False                 # greedy decoding
        )

    gen  = out.sequences[0, input_len:]    # (3,) 생성된 action token ids
    lps  = [F.log_softmax(s[0].float(), dim=-1)[t].item()
            for s, t in zip(out.scores, gen)]
    conf   = sum(lps)                      # Σ log P (joint log-probability)
    action = processor.decode_actions(
        out.sequences[:, input_len:], unnorm_key=UNNORM_KEY
    )["actions"][0]                        # (7,): [dx,dy,dz,droll,dpitch,dyaw,gripper]
    return action, conf, lps


MAX_STEPS = 300
history   = []

for step in range(MAX_STEPS):
    action, conf, lps = get_action_and_confidence(obs["agentview_image"], task_desc)

    # EEF state 기록 (스텝 실행 전)
    eef_pos = obs.get("robot0_eef_pos", None)
    eef_z   = float(eef_pos[2]) if eef_pos is not None else None

    # 환경 스텝 실행
    obs, reward, done, _ = env.step(action.tolist())

    history.append({
        "step":   step,
        "conf":   conf,
        "lps":    lps,
        "probs":  [np.exp(lp) for lp in lps],
        "eef_z":  eef_z,
        "action": action,
        "reward": float(reward),
        "image":  obs["agentview_image"].copy(),
    })

    eef_str = f"  EEF_z={eef_z:.3f}" if eef_z is not None else ""
    print(
        f"[{step:3d}] ΣlogP={conf:7.2f}  "
        f"P=[{np.exp(lps[0]):.1%},{np.exp(lps[1]):.1%},{np.exp(lps[2]):.1%}]"
        + eef_str
    )
    if done:
        print(f"에피소드 종료 step={step}  success={reward > 0}")
        break

env.close()
print(f"\n완료: {len(history)} steps, final reward={history[-1]['reward']}")

## Cell 5: 시각화 + 상관분석

핵심 질문: **EEF Z ↓ (gripper 내려감) 와 confidence ↑ 가 상관이 있는가?**

- `Pearson r(EEF_z, confidence) < 0` → gripper 내려갈수록 confidence 증가 = 가설 지지

In [ ]:
from scipy.stats import pearsonr, spearmanr

steps = [h["step"]     for h in history]
confs = [h["conf"]     for h in history]
p_t   = [h["probs"][0] for h in history]   # translation token P
p_r   = [h["probs"][1] for h in history]   # rotation token P
p_g   = [h["probs"][2] for h in history]   # gripper token P
eef_z = [h["eef_z"]    for h in history]
has_z = eef_z[0] is not None

def smooth(a, w=5):
    return np.convolve(a, np.ones(w) / w, mode="same")

ncols = 3 if has_z else 2
fig, axes = plt.subplots(2, ncols, figsize=(6 * ncols, 9))

# (1) Σ log P over time
ax = axes[0, 0]
ax.plot(steps, confs, alpha=0.3, color="steelblue", label="raw")
ax.plot(steps, smooth(confs), color="steelblue", lw=2, label="smoothed")
ax.set_title("Σ log P(action tokens) over steps", fontsize=11)
ax.set_xlabel("Step"); ax.set_ylabel("Σ log P")
ax.legend(); ax.grid(True)

# (2) Per-token probability
ax = axes[0, 1]
for vals, name, c in zip(
    [p_t, p_r, p_g], ["translation", "rotation", "gripper"],
    ["tomato", "seagreen", "royalblue"]
):
    ax.plot(steps, smooth(vals), color=c, lw=1.5, label=name)
ax.set_title("Per-token softmax probability", fontsize=11)
ax.set_xlabel("Step"); ax.set_ylabel("P")
ax.legend(); ax.grid(True)

# (3) EEF Z vs Confidence scatter (핵심 그래프)
if has_z:
    ax = axes[0, 2]
    sc = ax.scatter(eef_z, confs, c=steps, cmap="viridis", s=25, alpha=0.8)
    plt.colorbar(sc, ax=ax, label="Step")
    ax.set_title("Confidence vs. EEF Z\n(Z 낮을수록 gripper가 물체 근처)", fontsize=11)
    ax.set_xlabel("EEF Z (m)"); ax.set_ylabel("Σ log P")

# (4) EEF Z over time
ax = axes[1, 0]
if has_z:
    ax.plot(steps, eef_z, color="darkorange", lw=2)
    ax.set_title("EEF Z over time (↓ = gripper 내려감)", fontsize=11)
    ax.set_xlabel("Step"); ax.set_ylabel("Z (m)"); ax.grid(True)
else:
    ax.text(0.5, 0.5, "EEF state not available",
            ha="center", va="center", transform=ax.transAxes)

# (5) Gripper command
grips = [float(h["action"][-1]) for h in history]
ax = axes[1, 1 if has_z else 0]
ax.step(steps, grips, color="seagreen", where="mid", lw=2)
ax.set_title("Predicted gripper command", fontsize=11)
ax.set_xlabel("Step"); ax.set_ylabel("Gripper"); ax.grid(True)

# (6) 마지막 프레임
ax = axes[1, 2 if has_z else 1]
ax.imshow(history[-1]["image"][::-1, ::-1])
ax.set_title(f"Final frame (reward={history[-1]['reward']})", fontsize=10)
ax.axis("off")

plt.suptitle(f"SpatialVLA × LIBERO Confidence\nTask: '{task_desc}'", fontsize=13)
plt.tight_layout()
plt.show()

# 상관 분석
print("=" * 50)
print("상관 분석 결과")
print("=" * 50)

r_step, p_step = pearsonr(steps, confs)
print(f"step  vs Σ log P → Pearson r = {r_step:+.3f}  (p = {p_step:.4f})")

if has_z:
    r_z, p_z       = pearsonr(eef_z, confs)
    r_z_sp, p_z_sp = spearmanr(eef_z, confs)
    print(f"EEF_z vs Σ log P → Pearson  r = {r_z:+.3f}  (p = {p_z:.4f})")
    print(f"                   Spearman r = {r_z_sp:+.3f}  (p = {p_z_sp:.4f})")
    print()
    if r_z < -0.3 and p_z < 0.05:
        print("→ r_z < 0: gripper 내려갈수록 confidence ↑  [가설 지지]")
    elif r_z > 0.3 and p_z < 0.05:
        print("→ r_z > 0: gripper 올라갈수록 confidence ↑  [가설 반박]")
    else:
        print("→ 뚜렷한 상관 없음 (|r| < 0.3 또는 p ≥ 0.05)")